In [ ]:
import pandas as pd
import numpy
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime


In [ ]:
data = pd.read_csv("/content/HR_Performance_Dataset.xlsx - HR_Gender Diversity & Equality.csv")


In [ ]:
data = data.drop(columns=['Surname', 'Name'])
data = data.drop_duplicates(keep='last').reset_index(drop=True)


In [ ]:
data['Leave Date'].fillna('-1', inplace=True)


<ipython-input-574-c507f8041505>:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['Leave Date'].fillna('-1', inplace=True)


In [ ]:
data["Year"] = 0;
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 691 entries, 0 to 690
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Employee ID            691 non-null    int64 
 1   DOB                    691 non-null    object
 2   Emp Age                691 non-null    int64 
 3   Gender                 691 non-null    object
 4   Marital Status         691 non-null    object
 5   Branch                 691 non-null    object
 6   Hire Date              691 non-null    object
 7   Leave Date             691 non-null    object
 8   Leave Reason           691 non-null    object
 9   Status                 691 non-null    object
 10  Department             691 non-null    object
 11  Employee Satisfaction  691 non-null    int64 
 12  Annual Salary ($)      691 non-null    int64 
 13  Bonus ($)              691 non-null    int64 
 14  Total Compensation     691 non-null    int64 
 15  Job Title              

In [ ]:
data['Hire Date'] = pd.to_datetime(data['Hire Date'], format='%m/%d/%Y')
data['Leave Date'] = data['Leave Date'].apply(lambda x: pd.to_datetime(x, format='%m/%d/%Y', errors='coerce') if x != '-1' else pd.NaT)
for index, row in data.iterrows():
    # Chỉ xử lý các dòng có giá trị thực tế cho cả Hire Date và Leave Date
    if pd.notna(row['Leave Date']) and pd.notna(row['Hire Date']):
        if row['Leave Date'] < row['Hire Date']:
            # Đổi chỗ
            data.at[index, 'Hire Date'], data.at[index, 'Leave Date'] = row['Leave Date'], row['Hire Date']

# Chuyển lại các cột 'Hire Date' và 'Leave Date' về dạng chuỗi sau khi đổi chỗ
data['Hire Date'] = data['Hire Date'].dt.strftime('%m/%d/%Y')
data['Leave Date'] = data['Leave Date'].apply(lambda x: x.strftime('%m/%d/%Y') if pd.notna(x) else '-1')

In [ ]:
# xoa bo cac dong co bonus = 0
for employee_id, group in data.groupby('Employee ID'):
    if (len(group) == 2):
        if int(group.iloc[0]["Bonus ($)"]) == 0:
            data = data.drop(group.index[0])
data = data.reset_index(drop=True)


In [ ]:
for employee_id, group in data.groupby('Employee ID'):
    for i in range(len(group) - 1):  # Bắt đầu từ dòng cuối và không thay đổi dòng cuối
        if group.iloc[i]['Bonus ($)'] == group.iloc[i + 1]['Annual Salary ($)']:
            data.loc[group.index[i], 'Annual Salary ($)'] = (group.iloc[i]['Annual Salary ($)'] + group.iloc[i + 1]['Annual Salary ($)']) // 2
            data.loc[group.index[i], 'Bonus ($)'] = min(group.iloc[i]['Bonus ($)'], group.iloc[i + 1]['Bonus ($)'])
            data = data.drop(group.index[i + 1])
data = data.reset_index(drop=True)


In [ ]:
check = [0] * len(data)

for i in range(len(data) - 1, -1, -1):
    if len(str(data['Leave Date'][i])) >= 4 and check[i] == 0:  # Loại 2 và 3 (Có Leave Date)
        x = i
        ans = 0
        for j in range(i, -1, -1):
            if data['Employee ID'][j] == data['Employee ID'][i]:
                x = j
                if (len(str(data['Leave Date'][j])) > 5):
                    check[j] = 1
                    ans += 1
            else:
                break

        if ans <= 1:  # Loại 2: Có một leave date duy nhất
            oki = str(data["Hire Date"][x])
            s = oki.split("/")
            start = int(s[2])

            oki = str(data["Leave Date"][i])
            s = oki.split("/")
            end = int(s[2])

            for j in range(x, i + 1):
                data.loc[j, "Year"] = start
                check[j] = -1
                start += 1

            row_to_copy = data.iloc[i]
            new_rows = pd.DataFrame([row_to_copy] * (end - (start - 1)))
            data = pd.concat([data.iloc[:i + 1], new_rows, data.iloc[i + 1:]], ignore_index=True)

            for j in range(end - (start - 1)):
                data.loc[i + 1 + j, "Year"] = start
                start += 1
                check[i + 1 + j] = -1
        else:  # Loại 3: Nhân viên có nhiều Leave Date (Thu nhập thay đổi theo thời gian)
            start = int(data["Hire Date"][x].split("/")[2])
            j = x
            bb = i - x + 1
            for u in range(bb):
                if len(str(data['Leave Date'][j])) > 5:
                    end = int(data['Leave Date'][j].split("/")[2])
                    row_to_copy = data.iloc[j]
                    new_rows = pd.DataFrame([row_to_copy] * (end - (start)))
                    data = pd.concat([data.iloc[:j + 1], new_rows, data.iloc[j + 1:]], ignore_index=True)
                    for z in range(end - (start - 1)):
                        data.loc[x  + z, "Year"] = str(start) + ""
                        start += 1
                        check[x  + z] = -1
                        j = x + z;
                j += 1
                if data["Year"][j] == 0:
                      data.loc[j, "Year"] = start
                      check[j] = -1
                      start += 1

    # Loại 1: Chỉ có Hire Date và không có Leave Date (làm đến hết năm 2019)
    if data['Leave Date'][i] == '-1' and check[i] == 0:
        x = i
        for j in range(i, -1, -1):
            if data['Employee ID'][j] == data['Employee ID'][i]:
                x = j
            else:
                break

        oki = str(data["Hire Date"][x])
        s = oki.split("/")
        start = int(s[2])

        for j in range(x, i + 1):
            data.loc[j, "Year"] = start
            check[j] = -1
            start += 1

        row_to_copy = data.iloc[i]
        new_rows = pd.DataFrame([row_to_copy] * (2019 - (start - 1)))
        data = pd.concat([data.iloc[:i + 1], new_rows, data.iloc[i + 1:]], ignore_index=True)

        for j in range(2019 - (start - 1)):
            data.loc[i + 1 + j, "Year"] = start
            start += 1
            check[i + 1 + j] = -1

# Hiển thị kết quả sau khi thay đổi
print(data.head(20))

<ipython-input-579-facce09a148c>:49: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2016' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data.loc[x  + z, "Year"] = str(start) + ""


In [ ]:
data['Annual Salary ($)'] = pd.to_numeric(data['Annual Salary ($)'], errors='coerce')
data['Bonus ($)'] = pd.to_numeric(data['Bonus ($)'], errors='coerce')

# Đảm bảo cột 'Year' là kiểu int để tránh lỗi so sánh
data['Year'] = pd.to_numeric(data['Year'], errors='coerce')

# Kiểm tra các dòng có cùng Employee ID, Hire Date và Leave Date, nhưng có Annual Salary hoặc Bonus khác nhau
for employee_id in data['Employee ID'].unique():
    employee_data = data[data['Employee ID'] == employee_id]

    for hire_leave_group in employee_data.groupby(['Hire Date', 'Leave Date']):
        group = hire_leave_group[1]

        # Bỏ qua các dòng có Leave Date hoặc Hire Date không hợp lệ
        group_valid = group[(group['Leave Date'] != '-1') & (group['Leave Date'].notna()) & (group['Hire Date'].notna())]

        # Nếu có sự khác biệt về Annual Salary và Bonus trong nhóm hợp lệ
        if group_valid['Annual Salary ($)'].nunique() > 1 or group_valid['Bonus ($)'].nunique() > 1:
            avg_salary = group_valid['Annual Salary ($)'].mean()
            avg_bonus = group_valid['Bonus ($)'].mean()

            # Cập nhật Annual Salary và Bonus với giá trị trung bình
            data.loc[group_valid.index, 'Annual Salary ($)'] = avg_salary
            data.loc[group_valid.index, 'Bonus ($)'] = avg_bonus

            # Cập nhật Year cho tất cả các dòng trong nhóm với năm của dòng có year bé hơn
            min_year = group_valid['Year'].min()
            data.loc[group_valid.index, 'Year'] = min_year

            # Chỉ giữ lại dòng có Year ở giữa (chọn dòng có Year nhỏ nhất nếu có 2 dòng)
            median_year = group_valid['Year'].median()  # Dùng giá trị trung bình của Year làm chuẩn

            # Chỉ giữ lại dòng có Year bằng median_year
            group_valid = group_valid[group_valid['Year'] == median_year]

            # Xóa bỏ các dòng khác trong nhóm hợp lệ
            data = data.drop(data.loc[group_valid.index[1:]].index)

# Thay thế NaN thành 0 trước khi chuyển thành int
data['Annual Salary ($)'] = data['Annual Salary ($)'].fillna(0).round().astype('Int64')
data['Bonus ($)'] = data['Bonus ($)'].fillna(0).round().astype('Int64')

data['Annual Salary ($)'] = pd.to_numeric(data['Annual Salary ($)'], errors='coerce')
data['Bonus ($)'] = pd.to_numeric(data['Bonus ($)'], errors='coerce')

# Đảm bảo cột 'Year' là kiểu int để tránh lỗi so sánh
data['Year'] = pd.to_numeric(data['Year'], errors='coerce')

# Kiểm tra và xóa các dòng trùng lặp với cùng Employee ID, Hire Date, Leave Date và các giá trị giống nhau ở Annual Salary và Bonus



In [ ]:
print(data.head(10).to_string(index=False))
data.info()

 Employee ID       DOB  Emp Age Gender Marital Status     Branch  Hire Date Leave Date Leave Reason      Status Department  Employee Satisfaction  Annual Salary ($)  Bonus ($)  Total Compensation               Job Title Job Description Manager (Y/N) Performance  Year
       10000  8/9/1989       33      M       Divorced Montevideo 01/02/2014         -1 Not provided      Active Production                      4              41924       2096               44020 Production Technician I      Technician            No     Exceeds  2014
       10000  8/9/1989       33      M       Divorced Montevideo 01/02/2014         -1 Not provided      Active Production                      4              41924       2096               44020 Production Technician I      Technician            No     Exceeds  2015
       10000  8/9/1989       33      M       Divorced Montevideo 01/02/2014         -1 Not provided      Active Production                      4              41924       2096               44020 

In [ ]:
data = data[data['Year'] != 2020]

# Sử dụng .loc để đảm bảo không có cảnh báo khi tạo cột Total Compensation
data.loc[:, 'Total Compensation'] = data['Annual Salary ($)'] + data['Bonus ($)']

In [ ]:
data['Year'] = pd.to_numeric(data['Year'], errors='coerce')

# Duyệt từ dưới lên trên để thay đổi Status cho nhân viên nếu không phải dòng cuối mà có Status là Resignation
for employee_id, group in data.groupby('Employee ID'):
    # Duyệt qua nhóm nhân viên từ dưới lên trên
    for i in range(len(group) - 1):  # Bắt đầu từ dòng cuối và không thay đổi dòng cuối
        if group.iloc[i]['Status'] == 'Resignation':  # Nếu dòng này có Status là Resignation
            # Thay đổi 'Resignation' thành 'Active'
            data.loc[group.index[i], 'Status'] = 'Active'
            data.loc[group.index[i], 'Leave Reason'] = "Not provided"



In [ ]:
data['DOB'] = pd.to_datetime(data['DOB'], format='%m/%d/%Y')

# Tính tuổi vào năm 2019 và thay vào cột DOB
data['DOB'] = 2019 - data['DOB'].dt.year


In [ ]:
for employee_id, group in data.groupby('Employee ID'):
    for i in range(1, len(group)):
        if i != len(group) - 1 and data['Leave Date'][group.index[i]] != data['Leave Date'][group.index[i - 1]] and len(str(data['Leave Date'][group.index[i - 1]])) <= 5:
            data.loc[group.index[i], 'Leave Date'] = '-1'
        if group.iloc[i]['Hire Date'] == group.iloc[i - 1]['Hire Date'] and len(group.iloc[i - 1]['Hire Date']) >= 5:
            if group.iloc[i]['Leave Date'] != group.iloc[i - 1]['Leave Date'] and len(group.iloc[i - 1]['Leave Date']) >= 5:
                x = str(group.iloc[i - 1]['Leave Date'])
                li = x.split("/")
                if (int(li[1]) == 31):
                    li[1] = "01";
                    li[0] = str(int(li[0]) + 1)
                    if len(li[0]) == 1:
                        li[0] = "0" + li[0]
                else:
                    li[1] = str(int(li[1]) + 1)
                    if len(li[1]) == 1:
                        li[1] = "0" + li[1]
                tmp = "/".join(li)
                data.loc[group.index[i], 'Hire Date'] = tmp




In [ ]:
for employee_id, group in data.groupby('Employee ID'):
    for i in range(len(group) - 1):
        data.loc[group.index[i], 'Status'] = 'Active'
        data.loc[group.index[i], 'Leave Reason'] = "Not provided"


In [ ]:
for employee_id, group in data.groupby('Employee ID'):
    for i in range(len(group) - 1):
        if group['Year'].iloc[i] == group['Year'].iloc[i + 1]:
            data = data.drop(group.index[i])

data = data.reset_index(drop=True)

print(data.head(20))


    Employee ID  DOB  Emp Age Gender Marital Status        Branch   Hire Date  \
0         10000   30       33      M       Divorced    Montevideo  01/02/2014   
1         10000   30       33      M       Divorced    Montevideo  01/02/2014   
2         10000   30       33      M       Divorced    Montevideo  01/02/2014   
3         10000   30       33      M       Divorced    Montevideo  01/02/2014   
4         10000   30       33      M       Divorced    Montevideo  01/02/2014   
5         10000   30       33      M       Divorced    Montevideo  01/02/2014   
6         10001   64       67      M        Married          Lima  01/08/2014   
7         10001   64       67      M        Married          Lima  01/08/2014   
8         10001   64       67      M        Married          Lima  01/08/2014   
9         10001   64       67      M        Married          Lima  01/08/2014   
10        10001   64       67      M        Married          Lima  01/08/2014   
11        10002   33       3

In [ ]:
for employee_id, group in data.groupby('Employee ID'):
    if len(str(group.iloc[len(group) - 1]['Leave Date'])) > 5:
        x = str(group.iloc[len(group) - 1]['Leave Date'])
        li = x.split("/")
        ye = int(li[2])
        if int(group['Year'].iloc[len(group) - 1]) > ye:
            data.loc[group.index[len(group) - 1], 'Year'] = ye
        for i in range(len(group) - 1):
            if group['Year'].iloc[i] == ye:
                data = data.drop(group.index[i])
                break


# Reset lại index sau khi drop các dòng trùng lặp
data = data.reset_index(drop=True)

# Hiển thị kết quả sau khi thay đổi


In [ ]:
output_file_path = '/content/okila.csv'  # Đường dẫn và tên file
data.to_csv(output_file_path, index=False)

output_file_path

'/content/okila.csv'

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 942 entries, 0 to 941
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Employee ID            942 non-null    int64 
 1   DOB                    942 non-null    int32 
 2   Emp Age                942 non-null    int64 
 3   Gender                 942 non-null    object
 4   Marital Status         942 non-null    object
 5   Branch                 942 non-null    object
 6   Hire Date              942 non-null    object
 7   Leave Date             942 non-null    object
 8   Leave Reason           942 non-null    object
 9   Status                 942 non-null    object
 10  Department             942 non-null    object
 11  Employee Satisfaction  942 non-null    int64 
 12  Annual Salary ($)      942 non-null    Int64 
 13  Bonus ($)              942 non-null    Int64 
 14  Total Compensation     942 non-null    int64 
 15  Job Title              